# sub-16: resolving the disputed sweeps

**Where we are** (`sub16_stimulus_combined.ipynb`):
- The round trip shows the recovered chains are largely right: LE02 decoded from them reaches 8-9/9 (frame corr 0.53-0.55), against 0.38 with random directions.
- 2-3 sweeps per run remain disputed between the free chain and the chain with the generator structure (8 distinct directions + one half blank).
- Validation: the free chain makes 0°/45° confusions on A runs; the structured chain corrects them.

**This notebook** keeps the sweeps on which both chains agree and **enumerates every chain compatible with the generator** for the rest. Each hypothesis is then judged on 4 criteria that use different information:

| criterion | information used |
|---|---|
| S1 frame evidence | decoder + spatial pattern of the response (additive, as before) |
| S2 pRF model | full prediction (HRF, nonlinearity), median correlation over vertices |
| S3 global signal | mean of V1-V3: when the screen is blank (sensitive to the position of the blanks) |
| S4 travelling-wave latency | within each sweep, the order in which vertices respond vs the order in which the bar covers their pRF centre (sensitive to direction, insensitive to amplitude) |

plus **S5 round trip**: LE02 decoded from the hypotheses (the decoder never sees LE02, the truth is known).

**Validation first**: on A runs (sub-16 LE02, sub-13 LE01 and RE02), 3 consecutive sweeps are hidden, all compatible hypotheses are enumerated, and we check at which rank each criterion puts the truth. The decoders of the validation never see bars (rings/wedges only).

**Decision**: a disputed sweep is resolved when the criteria validated in step 1 and the round trip agree on the same hypothesis. Read only. Design matrices in `Desktop/fig_od/sub16_disputed/`.

In [1]:
import os, glob, math
from functools import lru_cache
import numpy as np, pandas as pd, nibabel as nb
import scipy.io as sio
import matplotlib.pyplot as plt

main = '/Users/uriel/disks/meso_shared/amblyo7T_prf'
pp = f'{main}/derivatives/pp_data'
out_dir = '/Users/uriel/Desktop/fig_od/sub16_combined'
os.makedirs(out_dir, exist_ok=True)
TR = 2.0
lags = [1, 2, 3, 4]                       # decoder: data at t+lag decodes the stimulus at t
model_lags = [2, 3]                       # pRF evidence: response pattern at t+lag
alphas = [1e1, 1e2, 1e3, 1e4, 1e5, 1e6]
dec_rois, dec_nmax = [1, 2, 3], 3000      # best variant of sub16_stimulus_decoder.ipynb
RW = ['RingsLeftEye_run-01', 'WedgesLeftEye_run-01', 'RingsRightEye_run-01', 'WedgesRightEye_run-01']
SUSPECTS = ['BarsLeftEye_run-01', 'BarsRightEye_run-01', 'BarsRightEye_run-02']
rng = np.random.default_rng(0)


/var/folders/60/q71xm4hx45x3db5d6mzl8r4h0000gn/T/ipykernel_7184/3008775802.py:3: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import numpy as np, pandas as pd, nibabel as nb


## Functions (same as the previous notebooks)

In [2]:
def build_vdm(mat_fn):
    """Same conversion as create_vdm_files.ipynb (50 x 50, transposed, padded)."""
    im = np.asarray(sio.loadmat(mat_fn, squeeze_me=True, struct_as_record=False)['s'].im, float)
    out = np.zeros((50, 50, im.shape[-1]))
    ii = np.floor(np.arange(50) * 102 / 50).astype(int)
    for t in range(im.shape[-1]):
        f = (im[..., t] * 255).astype(np.uint8).T
        sq = np.vstack([np.zeros((12, 102), np.uint8), f, np.zeros((13, 102), np.uint8)])
        out[..., t] = (sq > 5)[np.ix_(ii, ii)]
    return out

vdm = {eye: np.load(f'{main}/derivatives/vdm/task-BarsBarsRingsWedges{eye}_vdm.npy') for eye in ['LeftEye', 'RightEye']}
A = vdm['LeftEye'][..., :120]
B = build_vdm(f'{main}/sourcedata/C/jp/cond-1-stim-3.mat')
true_stim = {'A': A, 'B': B}


def run_stim(r, kind=None):
    """Known stimulus (50,50,120) of a run."""
    eye = 'LeftEye' if 'LeftEye' in r else 'RightEye'
    if r.startswith('Rings'): return vdm[eye][..., 240:360]
    if r.startswith('Wedges'): return vdm[eye][..., 360:480]
    return true_stim[kind]


def ds(st):
    """(50,50,T) -> (T, 625), 2x2 average."""
    return st.reshape(25, 2, 25, 2, -1).mean((1, 3)).reshape(625, -1).T

rr, cc = np.meshgrid(np.arange(50) - 24.5, np.arange(50) - 24.5, indexing='ij')
X_pix, Y_pix = cc, -rr


def frame_label(f):
    m = f > 0
    if m.sum() < 3: return None, np.nan
    best = None
    for a in [0, 45, 90, 135]:
        th = np.deg2rad(a); p = X_pix[m] * np.cos(th) + Y_pix[m] * np.sin(th)
        if best is None or p.std() < best[0]: best = (p.std(), a, p.mean())
    return best[1], best[2]


def segment(stim):
    lab = [frame_label(stim[..., t]) for t in range(stim.shape[-1])]
    ev, t = [], 0
    while t < len(lab):
        o, p = lab[t]; u = t + 1
        if o is None:
            while u < len(lab) and lab[u][0] is None: u += 1
            ev.append(('b', t, u - t))
        else:
            while u < len(lab) and lab[u][0] == o and u - t < 12: u += 1
            ev.append((f"{o}{'+' if lab[u - 1][1] > p else '-'}", t, u - t))
        t = u
    return ev

templates = {}
for stim in [A, B]:
    for k, s, l in segment(stim):
        if k != 'b' and l == 12 and k not in templates:
            templates[k] = stim[..., s:s + 12]
print(sorted(templates))


def tmpl(k, L):
    """Sweep k resampled to L frames, full resolution (50,50,L)."""
    return templates[k][..., np.round(np.linspace(0, 11, L)).astype(int)]


def chain_to_stim(chain):
    st = np.zeros((50, 50, 120))
    for k, t0, L in chain:
        if k == 'b': continue
        fr = tmpl(k, L)
        n = min(L, 120 - t0)
        st[..., t0:t0 + n] = fr[..., :n]
    return st


def chain_str(chain):
    return ' '.join(f'b{L}' if k == 'b' else (k if L == 12 else f'{k}/{L}') for k, t0, L in chain)

true_chain = {k: [(e[0], e[1], e[2]) for e in segment(v)] for k, v in true_stim.items()}
print('A:', chain_str(true_chain['A'])); print('B:', chain_str(true_chain['B']))

['0+', '0-', '135+', '135-', '45+', '45-', '90+', '90-']
A: b6 135+ 90+ b5 45- 0+ 135- 0- 45+ 90- 135- 135-/1
B: 0+ 135- 90- 45- 0- 135+ 90+ 45+ 0+ 135-


In [3]:
cache = {}
def load_subject(s):
    if s in cache: return cache[s]
    roi = np.concatenate([nb.load(glob.glob(f'{pp}/{s}/fsnative/rois/{s}_hemi-{h}_fmriprep_dct_*_rois-group-mmp.func.gii')[0]).darrays[0].data
                          for h in ['L', 'R']])
    prf = {eye: np.hstack([np.vstack([a.data for a in nb.load(
        f'{pp}/{s}/fsnative/prfmodel/prf_derivatives/{s}_task-RingsWedges{eye}_hemi-{h}_fmriprep_dct_psc_concat_prf-css_deriv.func.gii').darrays])
        for h in ['L', 'R']]) for eye in ['LeftEye', 'RightEye']}
    cache[s] = (roi, prf)
    return cache[s]

run_cache = {}
def load_run(s, r):
    if (s, r) in run_cache: return run_cache[(s, r)]
    hs = []
    for h in ['L', 'R']:
        d = np.vstack([a.data for a in nb.load(
            f'{pp}/{s}/fsnative/func/fmriprep_dct_psc/{s}_ses-01_task-{r}_hemi-{h}_space-fsnative_fmriprep_dct_psc_bold.func.gii').darrays])
        hs.append(d if d.shape[0] == 120 else d.T)
    run_cache[(s, r)] = np.hstack(hs).astype(np.float32)
    return run_cache[(s, r)]


def select_vertices(s, rois, n_max):
    roi, prf = load_subject(s)
    r2 = (prf['LeftEye'][0] + prf['RightEye'][0]) / 2
    ok = np.isfinite(r2) & (r2 > 0.1) & ((roi > 0) if rois is None else np.isin(roi, rois))
    idx = np.where(ok)[0]
    return idx[np.argsort(-r2[idx])[:n_max]]


def zc(a, axis=0):
    a = a - a.mean(axis, keepdims=True)
    return a / (a.std(axis, keepdims=True) + 1e-9)


def features(y):
    """(120, nv) -> (120, nv * n_lags): row t = data at t+lag."""
    y = zc(np.nan_to_num(y))
    cols = []
    for l in lags:
        z = np.zeros_like(y); z[:120 - l] = y[l:]
        cols.append(z)
    return np.hstack(cols) / np.sqrt(y.shape[1] * len(lags))

In [4]:
def frame_corr(D, S):
    """Median over non-blank frames of the correlation between decoded and true frames."""
    m = S.sum(1) > 0
    Dc = D[m] - D[m].mean(1, keepdims=True); Sc = S[m] - S[m].mean(1, keepdims=True)
    return float(np.median((Dc * Sc).sum(1) / (np.sqrt((Dc ** 2).sum(1) * (Sc ** 2).sum(1)) + 1e-9)))


def fit_predict(Xtr, Ytr, Xte, alpha):
    mu = Ytr.mean(0)
    K = Xtr @ Xtr.T
    W = np.linalg.solve(K + alpha * np.eye(len(K)), Ytr - mu)
    return Xte @ Xtr.T @ W + mu


def train_decoder(train):
    """train: list of (X, Y). Returns predict(X), chosen alpha, gain, LORO frame corr."""
    best = None
    for a in alphas:
        preds = [fit_predict(np.vstack([t[0] for j, t in enumerate(train) if j != i]),
                             np.vstack([t[1] for j, t in enumerate(train) if j != i]), train[i][0], a) for i in range(len(train))]
        sc = np.mean([frame_corr(p, t[1]) for p, t in zip(preds, train)])
        if best is None or sc > best[0]:
            best = (sc, a, preds)
    sc, a, preds = best
    P, Y = np.vstack(preds), np.vstack([t[1] for t in train])
    mu = Y.mean(0)
    gain = float(((P - mu) * (Y - mu)).sum() / ((Y - mu) ** 2).sum())   # shrinkage of the decoded amplitude
    Xtr, Ytr = np.vstack([t[0] for t in train]), np.vstack([t[1] for t in train])
    return (lambda X: fit_predict(Xtr, Ytr, X, a)), a, gain, sc, mu

In [5]:
pix = 2 * math.degrees(math.atan(50.6 / 2 / 242)) / 50
Xd, Yd = X_pix * pix, Y_pix * pix
t_ = np.arange(0, 32, TR)
g = lambda a: t_ ** (a - 1) * np.exp(-t_) / math.gamma(a)
H = g(6) - g(16) / 6; H /= H.sum()
N_fft = 256
Hf = np.fft.rfft(H, n=N_fft)


def conv_hrf(d):
    return np.fft.irfft(np.fft.rfft(d, n=N_fft, axis=0) * Hf[:, None], n=N_fft, axis=0)[:d.shape[0]]


def prf_score(stim, s, r):
    """Median correlation (V1-V3 vertices, pRFs of the eye) between data and pRF prediction, best delay -2..+2 TR."""
    roi, prf = load_subject(s)
    D = prf['LeftEye' if 'LeftEye' in r else 'RightEye']
    y = load_run(s, r)
    idx = np.where((D[0] > 0.2) & (D[1] < 6) & np.isin(roi, [1, 2, 3]) & np.isfinite(y).all(0) & (y.std(0) > 0))[0]
    x0, y0, sd, n = D[7][idx], D[8][idx], np.clip(D[4][idx], 0.2, None), np.clip(D[11][idx], 0.05, 1)
    G = np.exp(-((Xd.ravel()[:, None] - x0) ** 2 + (Yd.ravel()[:, None] - y0) ** 2) / (2 * sd ** 2))
    p = conv_hrf(np.clip(np.moveaxis(stim, -1, 0).reshape(120, -1) @ G, 0, None) ** n)
    yy = y[:, idx]
    best = -np.inf
    for sh in range(-2, 3):
        q = np.roll(p, sh, 0)
        if sh > 0: q[:sh] = p[0]
        elif sh < 0: q[sh:] = p[-1]
        best = max(best, float(np.median((zc(q) * zc(yy)).mean(0))))
    return best

In [6]:
keys = sorted(templates)
T = 120


def decoder_scores(D, gain, mu):
    """Frame scores from the decoded movie, relative to a blank frame: dict k -> (T, 12)."""
    Dc = D - mu
    blank_err = ((Dc + gain * mu) ** 2).sum(1)
    out = {}
    for k in keys:
        F = ds(tmpl(k, 12))
        err = ((Dc[:, None, :] - gain * (F - mu)[None]) ** 2).sum(2)
        out[k] = blank_err[:, None] - err
    return out


model_cache = {}
def model_scores(s, r):
    """Frame scores from the pRF model: spatial correlation between the response pattern at t+lag and the pRF drive of each template frame."""
    if (s, r) in model_cache: return model_cache[(s, r)]
    roi, prf = load_subject(s)
    Dp = prf['LeftEye' if 'LeftEye' in r else 'RightEye']
    y = load_run(s, r)
    idx = np.where((Dp[0] > 0.2) & (Dp[1] < 6) & np.isin(roi, [1, 2, 3]) & np.isfinite(y).all(0) & (y.std(0) > 0))[0]
    x0, y0, sd, n = Dp[7][idx], Dp[8][idx], np.clip(Dp[4][idx], 0.2, None), np.clip(Dp[11][idx], 0.05, 1)
    G = np.exp(-((Xd.ravel()[:, None] - x0) ** 2 + (Yd.ravel()[:, None] - y0) ** 2) / (2 * sd ** 2))
    Y = zc(y[:, idx])
    out = {}
    for k in keys:
        drv = np.clip(np.moveaxis(tmpl(k, 12), -1, 0).reshape(12, -1) @ G, 0, None) ** n
        Dz = zc(drv, axis=1)
        acc = 0
        for l in model_lags:
            Yl = np.zeros_like(Y); Yl[:T - l] = Y[l:]
            acc = acc + zc(Yl, axis=1) @ Dz.T / len(idx)
        out[k] = acc / len(model_lags)
    model_cache[(s, r)] = out
    return out


def normalise(S, center):
    allv = np.concatenate([S[k].ravel() for k in keys])
    c = np.median(allv) if center else 0.0
    sd = allv.std() + 1e-12
    return {k: (S[k] - c) / sd for k in keys}


def combine(*Ss):
    return {k: sum(S[k] for S in Ss) for k in keys}


def sweep_sums(S):
    """Score of a 12-TR sweep of k starting at t0 (cut at the end of the run)."""
    return {k: np.array([sum(S[k][t0 + i, i] for i in range(12) if t0 + i < T) for t0 in range(T)]) for k in keys}


In [7]:
def merge_blanks(chain):
    out = []
    for e in chain:
        if out and e[0] == 'b' and out[-1][0] == 'b':
            out[-1] = ('b', out[-1][1], out[-1][2] + e[2])
        else:
            out.append(e)
    return out


def dp_free(S):
    """Exact best chain, no constraint: sweeps (8 directions, 12 TR) and blanks (1-14 TR, score 0).
    Returns chain [(k, t0, L)], total score, and for each sweep the global margin over the best chain with another
    direction at the same place (max-marginals)."""
    sc = sweep_sums(S)
    fwd = np.full(T + 1, -np.inf); fwd[0] = 0; back = [None] * (T + 1)
    for t in range(T):
        if not np.isfinite(fwd[t]): continue
        for b in range(1, 15):
            u = min(T, t + b)
            if fwd[t] > fwd[u]: fwd[u] = fwd[t]; back[u] = (t, 'b', u - t)
        for k in keys:
            u = min(T, t + 12); v = fwd[t] + sc[k][t]
            if v > fwd[u]: fwd[u] = v; back[u] = (t, k, 12)
    bwd = np.full(T + 1, -np.inf); bwd[T] = 0
    for t in range(T - 1, -1, -1):
        bwd[t] = max(max(bwd[min(T, t + b)] for b in range(1, 15)), max(sc[k][t] + bwd[min(T, t + 12)] for k in keys))
    chain, u = [], T
    while u > 0:
        t, k, l = back[u]; chain.append((k, t, l)); u = t
    chain = merge_blanks(chain[::-1])
    opt = fwd[T]
    conf = []
    for k, t0, l in chain:
        if k == 'b': continue
        alt = max(fwd[t0] + sc[k2][t0] + bwd[min(T, t0 + 12)] for k2 in keys if k2 != k)
        conf.append((k, t0, opt - alt))
    return chain, opt, conf


def dp_perm(S, init_blank=range(3, 8), ins_blank=(5, 6)):
    """Exact best chain with the generator structure seen in A: initial blank, the 8 directions once each with one
    inserted half-cycle blank, then free sweeps until the end (at least 7 directions completed)."""
    sc = sweep_sums(S)
    full = (1 << len(keys)) - 1

    @lru_cache(maxsize=None)
    def V(t, mask, bl, pb):
        if t >= T:
            return (0.0 if bin(mask).count('1') >= 7 else -np.inf), None
        best = (-np.inf, None)
        for i, k in enumerate(keys):
            if mask == full or not (mask >> i) & 1:
                nm = mask if mask == full else mask | (1 << i)
                v = sc[k][t] + V(min(T, t + 12), nm, bl, False)[0]
                if v > best[0]: best = (v, (k, 12, nm, bl, False))
        if mask != full and not bl and mask and not pb:
            for b in ins_blank:
                v = V(min(T, t + b), mask, True, True)[0]
                if v > best[0]: best = (v, ('b', b, mask, True, True))
        return best

    starts = [(V(l, 0, False, True)[0], l) for l in init_blank]
    opt, l0 = max(starts)
    chain, t, state = [('b', 0, l0)], l0, (0, False, True)
    while t < T:
        _, ch = V(t, *state)
        k, l, nm, bl, pb = ch
        chain.append((k, t, min(l, T - t) if k == 'b' else 12))
        t, state = t + l, (nm, bl, pb)
    V.cache_clear()
    return chain, opt


def sweeps_of(chain):
    return [(k, t0) for k, t0, l in chain if k != 'b' and t0 + 12 <= T]


def n_correct(chain, true):
    """Number of full sweeps of the true chain found in the chain (same direction, start +-1 TR) / number of true sweeps."""
    est = sweeps_of(chain)
    tr = sweeps_of(true)
    ok = sum(any(k == k2 and abs(t - t2) <= 1 for k2, t2 in est) for k, t in tr)
    return f'{ok}/{len(tr)}'


def agreement(c1, c2):
    a, b = sweeps_of(c1), sweeps_of(c2)
    ok = sum(any(k == k2 and abs(t - t2) <= 1 for k2, t2 in b) for k, t in a)
    return f'{ok}/{len(a)}'


def cstr(chain):
    return ' '.join(f'b{l}' if k == 'b' else k for k, t0, l in chain)


In [8]:
calib = []   # (case, direction, start, margin, correct) for the sweeps of the validation runs
vert_cache = {}
def decode(s, r, train):
    """train: list of (run, stim (50,50,120)). Returns decoded movie, gain, mu, LORO frame corr."""
    if s not in vert_cache: vert_cache[s] = select_vertices(s, dec_rois, dec_nmax)
    idx = vert_cache[s]
    tr = [(features(load_run(s, rr)[:, idx]), ds(st)) for rr, st in train]
    pred, a, gain, loro, mu = train_decoder(tr)
    return pred(features(load_run(s, r)[:, idx])), gain, mu, loro


def analyse(s, r, train, true=None, label=''):
    D, gain, mu, loro = decode(s, r, train)
    Sd = normalise(decoder_scores(D, gain, mu), center=False)
    Sm = normalise(model_scores(s, r), center=True)
    Sc = combine(Sd, Sm)
    res = {'case': label, 'subject': s, 'run': r, 'LORO frame corr (train)': loro}
    chains = {}
    for name, S in [('decoder', Sd), ('pRF model', Sm), ('combined', Sc)]:
        ch, opt, conf = dp_free(S)
        chains[name] = ch
        res[f'{name}: chain'] = cstr(ch)
        if name == 'combined':
            res['combined: min margin'] = min(c_[2] for c_ in conf) if conf else np.nan
            res['combined: margins'] = ' '.join(f'{k}@{t0}:{m:.1f}' for k, t0, m in conf)
            res['_conf'] = conf
            if true is not None:
                tr = sweeps_of(true)
                for k, t0, m in conf:
                    if t0 + 12 <= T:
                        calib.append((label, k, t0, m, any(k == k2 and abs(t0 - t2) <= 1 for k2, t2 in tr)))
        if true is not None:
            res[f'{name}: correct'] = n_correct(ch, true)
    chp, optp = dp_perm(Sc)
    chains['combined + permutation'] = chp
    res['combined + permutation: chain'] = cstr(chp)
    res['free vs permutation (agreement)'] = agreement(chains['combined'], chp)
    if true is not None:
        res['combined + permutation: correct'] = n_correct(chp, true)
        res['frame corr decoded vs true'] = frame_corr(D, ds(chain_to_stim(true)))
    res['pRF score combined'] = prf_score(chain_to_stim(chains['combined']), s, r)
    res['pRF score combined + permutation'] = prf_score(chain_to_stim(chp), s, r)
    return res, chains, D


def show(rows, cols=None):
    df = pd.DataFrame([{k: v for k, v in r_.items() if not k.startswith('_')} for r_ in rows]).set_index('case')
    pd.set_option('display.max_colwidth', 120)
    print(df[cols].T.to_string() if cols else df.T.to_string())
    return df

stim_of = lambda r: run_stim(r)                    # known stimuli of rings / wedges


## Hypotheses and criteria

In [9]:
def chain_from_vdm(stim):
    return [(k, s, l) for k, s, l in segment(stim)]


def fixed_from(c1, c2):
    """Sweeps on which two chains agree (same direction, start +-1 TR); time taken from c2."""
    b = sweeps_of(c2)
    return [(k, t2) for k, t in sweeps_of(c1) for k2, t2 in b if k == k2 and abs(t - t2) <= 1]


def enumerate_chains(fixed, max_n=50000, init_blank=range(3, 8), ins_blank=(5, 6)):
    """All chains with the generator structure (initial blank, 8 distinct directions with one inserted half blank,
    then free sweeps) that contain every fixed sweep (direction, start +-1 TR)."""
    fixed = sorted(fixed, key=lambda z: z[1])
    out = []

    def rec(t, used, bl, pb, chain, nfix):
        if len(out) >= max_n: return
        # prune: a fixed sweep whose window has passed without being placed
        if nfix < len(fixed) and t > fixed[nfix][1] + 1: return
        if t >= T:
            if nfix == len(fixed) and len(used) >= 7: out.append(chain)
            return
        must = fixed[nfix] if (nfix < len(fixed) and abs(t - fixed[nfix][1]) <= 1) else None
        full = len(used) == 8
        for k in keys:
            if must is not None and k != must[0]: continue
            if not full and k in used: continue
            rec(t + 12, used if full else used | {k}, bl, False, chain + [(k, t, 12)], nfix + (must is not None))
        if must is None and not full and not bl and used and not pb:
            for b in ins_blank:
                rec(t + b, used, True, True, chain + [('b', t, min(b, T - t))], nfix)

    for l in init_blank:
        rec(l, frozenset(), False, True, [('b', 0, l)], 0)
    return out


In [10]:
class RunScorer:
    """Scores of a candidate chain for one run:
    S1 combined frame evidence (decoder + pRF patterns, additive), S2 pRF forward model (median correlation over vertices),
    S3 global signal (mean of V1-V3 vertices: blanks vs sweeps), S4 travelling-wave latency (order in which vertices
    respond within each sweep vs order in which the bar covers their pRF centre)."""

    def __init__(self, s, r, Sc):
        self.s, self.r = s, r
        self.sweep_sc = sweep_sums(Sc)
        roi, prf = load_subject(s)
        Dp = prf['LeftEye' if 'LeftEye' in r else 'RightEye']
        y = load_run(s, r)
        idx = np.where((Dp[0] > 0.2) & (Dp[1] < 6) & np.isin(roi, [1, 2, 3]) & np.isfinite(y).all(0) & (y.std(0) > 0))[0]
        x0, y0, sd, n = Dp[7][idx], Dp[8][idx], np.clip(Dp[4][idx], 0.2, None), np.clip(Dp[11][idx], 0.05, 1)
        G = np.exp(-((Xd.ravel()[:, None] - x0) ** 2 + (Yd.ravel()[:, None] - y0) ** 2) / (2 * sd ** 2))
        self.drv = {k: np.clip(np.moveaxis(tmpl(k, 12), -1, 0).reshape(12, -1) @ G, 0, None) ** n for k in keys}
        self.Y = zc(y[:, idx]).astype(np.float32)
        self.gY = zc(y[:, idx].mean(1)[:, None])[:, 0]
        # travelling-wave latency table
        good = Dp[0][idx] > 0.3
        ii = np.clip(np.round(24.5 - y0 / pix).astype(int), 0, 49)
        jj = np.clip(np.round(x0 / pix + 24.5).astype(int), 0, 49)
        self.lat = {}
        for k in keys:
            fr = tmpl(k, 12)[ii, jj, :] > 0                    # (nv, 12) frames covering each pRF centre
            cov = fr.sum(1) > 0
            cover_t = np.where(cov, (fr * np.arange(12)).sum(1) / np.maximum(fr.sum(1), 1), np.nan)
            v = good & cov
            tab = np.zeros(T)
            for t0 in range(T - 4):
                seg = self.Y[t0 + 2: t0 + 14, v]
                if len(seg) < 6: continue
                w = seg - seg.min(0)
                com = (w * np.arange(len(seg))[:, None]).sum(0) / (w.sum(0) + 1e-9)
                tab[t0] = np.corrcoef(com, cover_t[v])[0, 1]
            self.lat[k] = np.nan_to_num(tab)

    def scores(self, chain):
        s1 = sum(self.sweep_sc[k][t0] for k, t0, l in chain if k != 'b')
        s4 = sum(self.lat[k][t0] for k, t0, l in chain if k != 'b' and t0 < T)
        d = np.zeros_like(self.Y)
        for k, t0, l in chain:
            if k == 'b' or t0 >= T: continue
            n = min(12, T - t0)
            d[t0:t0 + n] = self.drv[k][:n]
        p = conv_hrf(d)
        best2, best3 = -np.inf, -np.inf
        gp = p.mean(1)
        for sh in range(-2, 3):
            q = np.roll(p, sh, 0); g = np.roll(gp, sh)
            if sh > 0: q[:sh] = p[0]; g[:sh] = gp[0]
            elif sh < 0: q[sh:] = p[-1]; g[sh:] = gp[-1]
            best2 = max(best2, float(np.median((zc(q) * self.Y).mean(0))))
            best3 = max(best3, float(np.corrcoef(g, self.gY)[0, 1]))
        return {'S1 frame evidence': s1, 'S2 pRF model': best2, 'S3 global signal': best3, 'S4 latency': s4}


crit = ['S1 frame evidence', 'S2 pRF model', 'S3 global signal', 'S4 latency']


def rank_hypotheses(scorer, hyps):
    df = pd.DataFrame([scorer.scores(h) for h in hyps])
    for c_ in crit:
        df[c_ + ' (z)'] = (df[c_] - df[c_].mean()) / (df[c_].std() + 1e-12)
    df['all (sum z)'] = df[[c_ + ' (z)' for c_ in crit]].sum(1)
    df['chain'] = [cstr(h) for h in hyps]
    return df


## Step 1 — validation: hide 3 sweeps of a known A run, rank the truth among all compatible hypotheses

In [11]:
val_cases = [('sub-16', 'BarsLeftEye_run-02'), ('sub-13', 'BarsLeftEye_run-01'), ('sub-13', 'BarsRightEye_run-02')]
windows = [(0, 3), (2, 5), (4, 7), (6, 9)]          # indices of the hidden sweeps among the 9 full sweeps of A
val_rows = []
for s, r in val_cases:
    D, gain, mu, _ = decode(s, r, [(rr, run_stim(rr)) for rr in RW])      # decoder never trained on bars
    Sc = combine(normalise(decoder_scores(D, gain, mu), center=False), normalise(model_scores(s, r), center=True))
    sc_ = RunScorer(s, r, Sc)
    sw = sweeps_of(true_chain['A'])
    for a, b in windows:
        fixed = [x for i, x in enumerate(sw) if not (a <= i < b)]
        hyps = enumerate_chains(fixed)
        truth = [i for i, h in enumerate(hyps) if n_correct(h, true_chain['A']) == f'{len(sw)}/{len(sw)}']
        df = rank_hypotheses(sc_, hyps)
        row = {'run': f'{s} {r}', 'hidden sweeps': f'{a + 1}-{b}', 'n hypotheses': len(hyps), 'truth in set': bool(truth)}
        for c_ in crit + ['all (sum z)']:
            rk = df[c_].rank(ascending=False, method='min')
            row[f'rank of truth: {c_}'] = int(rk[truth].min()) if truth else np.nan
        val_rows.append(row)
        print(s, r, a, b, len(hyps), flush=True)
val = pd.DataFrame(val_rows)
print(val.to_string())
print('\nNumber of tests where the truth is ranked 1st:')
print((val[[c_ for c_ in val.columns if c_.startswith('rank of truth')]] == 1).sum().to_string())


sub-16 BarsLeftEye_run-02 0 3 612
sub-16 BarsLeftEye_run-02 2 5 624
sub-16 BarsLeftEye_run-02 4 7 156
sub-16 BarsLeftEye_run-02 6 9 416
sub-13 BarsLeftEye_run-01 0 3 612
sub-13 BarsLeftEye_run-01 2 5 624
sub-13 BarsLeftEye_run-01 4 7 156
sub-13 BarsLeftEye_run-01 6 9 416
sub-13 BarsRightEye_run-02 0 3 612
sub-13 BarsRightEye_run-02 2 5 624
sub-13 BarsRightEye_run-02 4 7 156
sub-13 BarsRightEye_run-02 6 9 416
                           run hidden sweeps  n hypotheses  truth in set  rank of truth: S1 frame evidence  rank of truth: S2 pRF model  rank of truth: S3 global signal  rank of truth: S4 latency  rank of truth: all (sum z)
0    sub-16 BarsLeftEye_run-02           1-3           612          True                                 1                            9                              107                          1                           1
1    sub-16 BarsLeftEye_run-02           3-5           624          True                                 1                            1     

## Step 2 — suspect runs: all hypotheses compatible with the agreed sweeps

In [12]:
comb_dir = '/Users/uriel/Desktop/fig_od/sub16_combined'
best_by = {}
res_sus = {}
for r in SUSPECTS:
    c_free = chain_from_vdm(np.load(f'{comb_dir}/sub-16_{r}_combined_vdm.npy'))
    c_perm = chain_from_vdm(np.load(f'{comb_dir}/sub-16_{r}_combined_permutation_vdm.npy'))
    fixed = fixed_from(c_free, c_perm)
    hyps = enumerate_chains(fixed)
    # decoder: RW + LE02 + the two other suspect runs with their free chain (as in pass 2)
    train = [(rr, run_stim(rr)) for rr in RW] + [('BarsLeftEye_run-02', A)] + \
            [(o, np.load(f'{comb_dir}/sub-16_{o}_combined_vdm.npy')) for o in SUSPECTS if o != r]
    D, gain, mu, _ = decode('sub-16', r, train)
    Sc = combine(normalise(decoder_scores(D, gain, mu), center=False), normalise(model_scores('sub-16', r), center=True))
    sc_ = RunScorer('sub-16', r, Sc)
    df = rank_hypotheses(sc_, hyps + [c_free])
    df['note'] = [''] * len(hyps) + ['free chain (no structure)']
    res_sus[r] = (df, hyps + [c_free])
    print(f'\n=== {r} | fixed sweeps (free = permutation): {" ".join(f"{k}@{t}" for k, t in fixed)} | {len(hyps)} hypotheses')
    show_cols = crit + ['all (sum z)', 'chain', 'note']
    for c_ in ['all (sum z)'] + crit:
        top = df.sort_values(c_, ascending=False).head(3)
        print(f'-- top 3 by {c_}')
        print(top[show_cols].round(3).to_string())
    best_by[r] = {c_: df[c_].idxmax() for c_ in crit + ['all (sum z)']}
    print('agreement of the criteria on the best hypothesis:', len(set(best_by[r].values())) == 1,
          {c_: int(v) for c_, v in best_by[r].items()})



=== BarsLeftEye_run-01 | fixed sweeps (free = permutation): 90+@5 45-@59 135+@71 90-@83 135-@95 0-@107 | 792 hypotheses
-- top 3 by all (sum z)
     S1 frame evidence  S2 pRF model  S3 global signal  S4 latency  all (sum z)                                            chain                       note
792            210.099         0.363             0.392       5.268        8.825  b5 90+ 0+ b5 0+ b1 135- 45- 135+ 90- 135- 0- b1  free chain (no structure)
445            142.276         0.316             0.418       4.669        5.577     b5 90+ 45+ 0- b6 0+ 45- 135+ 90- 135- 0- 45-                           
442            142.276         0.316             0.418       4.669        5.577    b5 90+ 45+ 0- b6 0+ 45- 135+ 90- 135- 0- 135+                           
-- top 3 by S1 frame evidence
     S1 frame evidence  S2 pRF model  S3 global signal  S4 latency  all (sum z)                                            chain                       note
792            210.099         0.363         

## Step 3 — round trip for the best hypotheses (LE02 decoded from them, never seen by the decoder)

In [13]:
# round trip: LE02 decoded by a decoder trained on RW + the 3 suspect runs; one run at a time varies over its top hypotheses
# (others fixed to their best 'all (sum z)' hypothesis). The decoder never sees LE02.
cur = {r: res_sus[r][1][int(res_sus[r][0]['all (sum z)'].idxmax())] for r in SUSPECTS}
rt_rows = []
for r in SUSPECTS:
    df, hyps = res_sus[r]
    cand = list(dict.fromkeys(list(df.sort_values('all (sum z)', ascending=False).index[:4]) +
                              [int(df[c_].idxmax()) for c_ in crit] + [len(hyps) - 1]))
    for i in cand:
        chains = dict(cur); chains[r] = hyps[i]
        train = [(rr, run_stim(rr)) for rr in RW] + [(o, chain_to_stim(chains[o])) for o in SUSPECTS]
        D, gain, mu, _ = decode('sub-16', 'BarsLeftEye_run-02', train)
        Sd = normalise(decoder_scores(D, gain, mu), center=False)
        ch, _, _ = dp_free(Sd)
        rt_rows.append({'varied run': r, 'hypothesis': i, 'chain': cstr(hyps[i]), 'note': df.loc[i, 'note'],
                        'all (sum z)': df.loc[i, 'all (sum z)'],
                        'LE02 frame corr vs A': frame_corr(D, ds(A)), 'LE02 sweeps correct (decoder)': n_correct(ch, true_chain['A'])})
    print(r, 'done', flush=True)
rt = pd.DataFrame(rt_rows)
pd.set_option('display.max_colwidth', 120)
print(rt.round(4).to_string())


BarsLeftEye_run-01 done
BarsRightEye_run-01 done
BarsRightEye_run-02 done
             varied run  hypothesis                                            chain                       note  all (sum z)  LE02 frame corr vs A LE02 sweeps correct (decoder)
0    BarsLeftEye_run-01         792  b5 90+ 0+ b5 0+ b1 135- 45- 135+ 90- 135- 0- b1  free chain (no structure)       8.8254                0.5380                           8/9
1    BarsLeftEye_run-01         445     b5 90+ 45+ 0- b6 0+ 45- 135+ 90- 135- 0- 45-                                  5.5768                0.5279                           8/9
2    BarsLeftEye_run-01         442    b5 90+ 45+ 0- b6 0+ 45- 135+ 90- 135- 0- 135+                                  5.5768                0.5294                           8/9
3    BarsLeftEye_run-01         444     b5 90+ 45+ 0- b6 0+ 45- 135+ 90- 135- 0- 45+                                  5.5756                0.5278                           8/9
4    BarsLeftEye_run-01         458    b5

## Step 4 — retained chain per run

In [14]:
out2 = '/Users/uriel/Desktop/fig_od/sub16_disputed'
os.makedirs(out2, exist_ok=True)
fin_rows = []
for r in SUSPECTS:
    df, hyps = res_sus[r]
    sub = rt[rt['varied run'] == r]
    i_rt = int(sub.loc[sub['LE02 frame corr vs A'].idxmax(), 'hypothesis'])
    i_all = int(df['all (sum z)'].idxmax())
    votes = [int(df[c_].idxmax()) for c_ in crit] + [i_all, i_rt]
    i_best = max(set(votes), key=votes.count)
    fin_rows.append({'run': r, 'best (sum z)': cstr(hyps[i_all]), 'best (round trip)': cstr(hyps[i_rt]),
                     'votes for the retained hypothesis (out of 6)': votes.count(i_best),
                     'retained': cstr(hyps[i_best]), 'is the free chain': df.loc[i_best, 'note'] != ''})
    np.save(f'{out2}/sub-16_{r}_retained_vdm.npy', chain_to_stim(hyps[i_best]))
fin = pd.DataFrame(fin_rows).set_index('run')
print(fin.T.to_string())
print('\nSaved in', out2)


run                                                                        BarsLeftEye_run-01                              BarsRightEye_run-01                           BarsRightEye_run-02
best (sum z)                                  b5 90+ 0+ b5 0+ b1 135- 45- 135+ 90- 135- 0- b1   b5 90- 0+ 135- 45- 135+ 90+ b5 0- 45+ 90- 135+  b5 135- 0- 0+ 135+ 45- b6 135- 90+ 90- 0+ b1
best (round trip)                             b5 90+ 0+ b5 0+ b1 135- 45- 135+ 90- 135- 0- b1  b5 135- 0+ 135- 45- 135+ 90+ b5 0- 0+ b1 90- b1      b6 135- 0- 0+ 135+ 45- b6 45+ 90+ 90- 0+
votes for the retained hypothesis (out of 6)                                                5                                                3                                             4
retained                                      b5 90+ 0+ b5 0+ b1 135- 45- 135+ 90- 135- 0- b1  b5 135- 0+ 135- 45- 135+ 90+ b5 0- 0+ b1 90- b1  b5 135- 0- 0+ 135+ 45- b6 135- 90+ 90- 0+ b1
is the free chain                                      

## How to read

- **Step 1**: which criteria rank the truth 1st (out of 12 tests). Trust only those criteria (and their sum) for step 2.
- **Step 2**: for each suspect run, whether the validated criteria point to the same hypothesis. The free chain (no structure) is added: if it wins on the validated criteria, the generator structure is probably not the right one for these runs.
- **Step 3**: the hypothesis that best teaches the decoder to read LE02 is the most likely one (non-circular criterion: the truth of LE02 is known).
- **Step 4**: retained chain = the most voted hypothesis among the 4 criteria, their sum and the round trip. It is solid if the validated criteria and the round trip vote for it.